# Arabic Sentiment Analysis

A reproducible study of Arabic sentiment classification on the ArSarcasm-v2 dataset.

## Notebook roadmap

1. Setup and configuration
2. Dataset loading and validation
3. Exploratory data analysis and splitting
4. Arabic text preprocessing
5. Feature representations
6. Model development and tuning
7. Preprocessing ablation
8. Data augmentation
9. Class-imbalance handling
10. Comparative evaluation and error analysis

## 1. Setup and configuration

This section loads the shared experiment configuration, fixes sources of randomness, and selects the available compute device.

In [ ]:
from __future__ import annotations

import platform
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml

In [ ]:
def find_project_root(start: Path) -> Path:
    """Find the repository root from a notebook or project-root kernel."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the repository root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
CONFIG_PATH = PROJECT_ROOT / "configs" / "default.yaml"

with CONFIG_PATH.open(encoding="utf-8") as config_file:
    CONFIG = yaml.safe_load(config_file)

PATHS = {
    name: PROJECT_ROOT / relative_path
    for name, relative_path in CONFIG["paths"].items()
}

CONFIG_PATH.relative_to(PROJECT_ROOT)

In [ ]:
SEED = int(CONFIG["project"]["seed"])
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if CONFIG["runtime"]["deterministic"]:
    torch.use_deterministic_algorithms(True, warn_only=True)

device_preference = CONFIG["runtime"]["device"]
if device_preference == "auto":
    device_name = "mps" if torch.backends.mps.is_available() else "cpu"
else:
    device_name = device_preference

DEVICE = torch.device(device_name)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_theme(context="notebook", style="whitegrid")

In [ ]:
pd.DataFrame(
    {
        "value": [
            CONFIG["project"]["name"],
            platform.python_version(),
            torch.__version__,
            str(DEVICE),
            SEED,
        ]
    },
    index=["project", "python", "pytorch", "device", "seed"],
)